# 06.8 — L2 inside the loss kernel (and why the port doesn't do it)

MATLAB regularizes weights by adding `L2Factor · w` to the **gradient** — as if an L2 penalty lived inside the loss. On plain SGD that's identical to "weight decay." On **Adam it is not** — and this is one of the very few places the port *deliberately diverges* from MATLAB rather than matching it. The project uses `torch.optim.AdamW` (decoupled weight decay), forwarding `l2_factor → weight_decay`, because grad-side L2 on Adam is mathematically the *wrong* thing. This notebook shows exactly why.

This notebook covers:

* L2 regularization: the penalty, and its gradient `λ·w`.
* Grad-side (coupled) vs decoupled weight decay — the two ways to apply it.
* Why they're identical on SGD but different on Adam (the AdamW insight).
* What the port does: `AdamW(weight_decay=l2_factor)`, a deliberate improvement.

**Prerequisites:** [02.7 optimizers](../02_numpy_and_pytorch_basics/02.7_optimizers.ipynb), [04.8 weight initialization](../04_architecture/04.8_weight_initialization.ipynb).

## Section 1 — What MATLAB does

MATLAB's `trainingOptions('adam', 'L2Regularization', lambda)` adds the L2 term to the gradient *before* the optimizer's update:

```matlab
opts = trainingOptions('adam', 'L2Regularization', 1e-4, ...);
% Internally, for each parameter w:
%   grad_total = grad_data + L2Factor * w      % <- L2 folded into the gradient
%   w = adamUpdate(w, grad_total)              % <- Adam then processes the SUM
```

The L2 penalty is **coupled** to the gradient — Adam's adaptive machinery sees `grad_data + λ·w` as one quantity and rescales the whole thing. That coupling is the crux: it means the effective decay on each weight depends on that weight's gradient history, which is *not* what "decay all weights by λ" is supposed to mean. The port does something different, and §2 shows why that's correct.

## Section 2 — The Python concepts you need

### 2.1 — L2 regularization, and its gradient

L2 regularization adds a penalty for large weights to the objective:

$$\mathcal{L}_{total} = \mathcal{L}_{data} + \frac{\lambda}{2}\lVert w \rVert^2$$

The gradient of that penalty term is simply `λ·w`. So there are two mechanically-different ways to apply it:

* **Coupled (grad-side):** add `λ·w` to the gradient, let the optimizer process the sum. This is MATLAB's `L2Regularization`.
* **Decoupled (weight decay):** leave the gradient alone; separately shrink each weight by `lr·λ·w` after the optimizer step. This is `AdamW`.

They *sound* equivalent — and for SGD they are. For Adam they are not.

### 2.2 — On SGD, the two are identical

In [ ]:
import torch

def train_toy(opt_name, mode, steps=15, wd=0.1, lr=0.02, start=2.0):
    """Minimize 0.5*w^2 (data grad = w) under coupled vs decoupled L2."""
    w = torch.nn.Parameter(torch.tensor([start]))
    decoupled_wd = wd if mode == "decoupled" else 0.0
    Opt = torch.optim.SGD if opt_name == "SGD" else torch.optim.AdamW
    opt = Opt([w], lr=lr, weight_decay=decoupled_wd)
    traj = [w.item()]
    for _ in range(steps):
        opt.zero_grad()
        loss = 0.5 * (w ** 2).sum()          # data loss; gradient = w
        loss.backward()
        if mode == "coupled":
            w.grad = w.grad + wd * w.detach()  # grad-side L2 (MATLAB style)
        opt.step()
        traj.append(w.item())
    return traj

sgd_coupled   = train_toy("SGD", "coupled")
sgd_decoupled = train_toy("SGD", "decoupled")
print(f"SGD coupled  final w: {sgd_coupled[-1]:.6f}")
print(f"SGD decoupled final w: {sgd_decoupled[-1]:.6f}")
print("→ IDENTICAL. On SGD, adding λ·w to the grad == decaying w by lr·λ. Same update.")

On SGD the update is `w ← w − lr·(grad + λ·w) = w − lr·grad − lr·λ·w`, which is *exactly* "take the data step, then shrink w by `lr·λ·w`." Coupled and decoupled coincide. This is why the distinction went unnoticed for years — everyone learned L2 on SGD, where it doesn't matter.

### 2.3 — On Adam, they diverge (the AdamW insight)

In [ ]:
adam_coupled   = train_toy("AdamW", "coupled")     # MATLAB-style: L2 in the gradient
adamw_decoupled = train_toy("AdamW", "decoupled")   # AdamW-style: decoupled decay
print(f"Adam  coupled (L2 in grad) final w: {adam_coupled[-1]:.6f}")
print(f"AdamW decoupled (weight_decay) final w: {adamw_decoupled[-1]:.6f}")
print(f"difference: {abs(adam_coupled[-1] - adamw_decoupled[-1]):.6f}  → NOT the same optimizer!")

They differ. Here's why: Adam divides every gradient by `√v` (a per-parameter running estimate of gradient magnitude, [02.7 §2.3](../02_numpy_and_pytorch_basics/02.7_optimizers.ipynb)). When you fold `λ·w` *into* the gradient (coupled), that L2 term gets divided by `√v` too — so each weight's effective decay becomes **entangled with its own gradient history** rather than being a clean, uniform `λ`. The regularization strength turns into an accident of gradient magnitude. AdamW **decouples** the decay: it applies `lr·λ·w` directly to the weight, untouched by `√v`, so every weight decays at the honest rate λ. Loshchilov & Hutter (2019) showed this decoupled form is the correct one and that it generalizes better.

### 2.4 — See the divergence

In [ ]:
import matplotlib.pyplot as plt

fig, (axS, axA) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axS.plot(sgd_coupled, "o-", label="coupled (grad+L2)", markersize=5)
axS.plot(sgd_decoupled, "x--", label="decoupled (weight_decay)", markersize=7)
axS.set_title("SGD — the two curves OVERLAP"); axS.set_xlabel("step"); axS.set_ylabel("weight")
axS.legend()
axA.plot(adam_coupled, "o-", label="Adam coupled (grad+L2 / MATLAB)", markersize=5)
axA.plot(adamw_decoupled, "x--", label="AdamW decoupled (the port)", markersize=7)
axA.set_title("Adam — the two curves SEPARATE"); axA.set_xlabel("step"); axA.legend()
plt.tight_layout(); plt.show()
print("SGD: coupled and decoupled trace the same path. Adam: they split — different optimizers.")

### 2.5 — What the port does, and why it's a *deliberate* divergence

The port uses `torch.optim.AdamW` and forwards MATLAB's `L2Factor` straight into `weight_decay`:

```python
optimizer = torch.optim.AdamW(params, lr=..., weight_decay=cfg.l2_factor)  # decoupled
```

This is one of the rare spots where the port **intentionally does not match MATLAB**. Everywhere else, MATLAB is the behavioral reference and parity is the goal. Here, MATLAB's grad-side L2 on Adam is a known-suboptimal recipe (Adam + coupled L2), and blindly reproducing it would *bake in a defect*. AdamW is the mathematically-cleaner, better-generalizing form — so the port adopts it and documents the choice. The lesson: parity with a reference is the default, but not when the reference is doing something the field has since improved on. (For `Optimizer='SGDM'`, coupled and decoupled coincide anyway (§2.2), so `SGD(weight_decay=...)` is exact parity there.)

Note the title of this notebook is a slight trap of its own: there is **no** L2 term computed inside the loss kernel in this codebase — no `weight_decay.py`, nothing added to the loss sum ([06.1](06.1_multi_task_losses_overview.ipynb)). The regularization lives entirely in the optimizer. "L2 inside the loss kernel" is how MATLAB *thinks* about it (grad-side); the port moves it out to the optimizer where it belongs.

## Section 3 — The `neural_data_decoding` implementation

The wiring in `cli.py` — `l2_factor` becomes AdamW's `weight_decay`:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/cli.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "weight_decay=float(cfg.l2_factor)" in line)
for k in range(i - 4, i + 2):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `weight_decay=float(cfg.l2_factor)` — MATLAB's `L2Factor` (default `1e-4` in `configs/base.yaml`) flows directly into the optimizer's decoupled `weight_decay`. No L2 term is ever added to the loss.
* The optimizer factory is chosen by config (`ADAM → AdamW`, `SGDM → SGD`) — the [04.1](../04_architecture/04.1_architecture_string_dispatcher.ipynb) string-dispatch pattern again ([02.7 §3](../02_numpy_and_pytorch_basics/02.7_optimizers.ipynb) covers the factory).
* `build_optimizer_with_module_groups` (`training/freezing.py`) threads the same `weight_decay` through the per-module parameter groups used by the freeze curriculum ([05.4](../05_training_loop/05.4_learning_rate_scheduling.ipynb), [Module 07](../07_dynamic_curriculum/)) — so decay is consistent across frozen and live groups.
* Rebuilding the optimizer (e.g. at a Stage 1→2 handoff, [05.6](../05_training_loop/05.6_the_two_stage_lifecycle.ipynb)) resets AdamW's moments but re-applies the same `weight_decay` — the regularization is a property of the optimizer config, re-established each build.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the SGD equivalence, exactly

Confirm coupled and decoupled L2 on SGD agree to floating-point precision at *every* step, not just the end.

In [ ]:
# Reveal:
import torch
maxdiff = max(abs(a - b) for a, b in zip(sgd_coupled, sgd_decoupled))
print(f"max per-step difference (SGD coupled vs decoupled): {maxdiff:.2e}")
print("→ ~0: on SGD they are the same update at every step.")

### Exercise 4.2 — coupling entangles decay with gradient scale

Run two weights with *different* data-gradient magnitudes under the same λ. Under **decoupled** AdamW they shrink by the *identical* fraction (decay is uniform λ). Under **coupled** Adam they shrink by *different* fractions — the `√v` division has leaked each weight's gradient scale into its regularization.

In [ ]:
# Reveal:
def final_shrink(data_grad_scale, mode):
    # Same L2 (wd=0.1) but different data-gradient magnitudes; measure how much
    # the weight shrank from its start under coupled vs decoupled Adam.
    w = torch.nn.Parameter(torch.tensor([1.0]))
    wd = 0.1
    opt = torch.optim.AdamW([w], lr=0.01, weight_decay=(wd if mode == "decoupled" else 0.0))
    for _ in range(40):
        opt.zero_grad()
        (data_grad_scale * w).sum().backward()   # data gradient ∝ scale
        if mode == "coupled":
            w.grad = w.grad + wd * w.detach()
        opt.step()
    return 1.0 - w.item()

dec_small, dec_large = final_shrink(0.1, "decoupled"), final_shrink(5.0, "decoupled")
cpl_small, cpl_large = final_shrink(0.1, "coupled"),   final_shrink(5.0, "coupled")
print(f"decoupled AdamW — small-grad shrank {dec_small:.4f}, large-grad {dec_large:.4f}  "
      f"→ identical? {abs(dec_small - dec_large) < 1e-6}")
print(f"coupled Adam    — small-grad shrank {cpl_small:.4f}, large-grad {cpl_large:.4f}  "
      f"→ identical? {abs(cpl_small - cpl_large) < 1e-6}")
print("→ decoupled: gradient scale does NOT touch decay. coupled: it does — regularization is entangled.")

## Section 5 — Common errors

### Reproducing MATLAB's Adam + grad-side L2 for "parity"

Don't. It's the one place the port intentionally diverges (§2.5). Adding `λ·w` to the gradient under Adam gives you the *worse* optimizer. Use `AdamW(weight_decay=...)`.

### Setting `weight_decay` on `torch.optim.Adam` (not AdamW)

`torch.optim.Adam`'s `weight_decay` is the **coupled** form — it adds L2 to the gradient internally, the MATLAB behavior. Only `torch.optim.AdamW` is decoupled. The class name is the difference; the port uses `AdamW` for exactly this reason.

### Also adding an L2 term to the loss sum

Then you're regularizing *twice* — once in the loss, once in the optimizer. There is no L2 in this project's loss kernel ([06.1](06.1_multi_task_losses_overview.ipynb)); it lives only in `weight_decay`. Adding your own doubles it.

### `l2_factor` set but decay seems absent

Check the optimizer actually received `weight_decay` — a hand-built optimizer that dropped the kwarg regularizes nothing. And confirm it's `AdamW`, not `Adam`, if you expect decoupled behavior.

### Weight decay applied to biases / norm parameters

Standard practice often excludes biases and normalization scales from decay. If a run over-regularizes, check whether the parameter groups ([05.4](../05_training_loop/05.4_learning_rate_scheduling.ipynb)) apply `weight_decay` indiscriminately — MATLAB's `L2Factor` and the port's default both decay all params, but this is a knob worth knowing.

## Section 6 — Further reading

- [Decoupled Weight Decay Regularization (Loshchilov & Hutter, 2019)](https://arxiv.org/abs/1711.05101) — the paper that introduced AdamW and named this exact distinction.
- [02.7 optimizers](../02_numpy_and_pytorch_basics/02.7_optimizers.ipynb) — where Adam's `√v` adaptive rescaling (the cause of the divergence) is built up.
- [`torch.optim.AdamW` docs](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html) — note the explicit "decoupled weight decay" in the description.

Next up: **[06.9 per-batch prior correction](06.9_per_batch_prior_correction.ipynb)** — the `want_batch_correction` flag in the confidence routing.